In [1]:
from itertools import product
import io
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

In [2]:
def build_dhis2_metadata_map(session: requests.Session, base_url: str) -> dict[str, str]:
    meta_map = {}
    endpoints = {
        "trackedEntityAttributes": f"{base_url}/api/trackedEntityAttributes.json?fields=id,displayName&paging=false",
        "dataElements": f"{base_url}/api/dataElements.json?fields=id,displayName&paging=false",
        "programStages": f"{base_url}/api/programStages.json?fields=id,displayName&paging=false",
        "programs": f"{base_url}/api/programs.json?fields=id,displayName&paging=false",
        "organisationUnits": f"{base_url}/api/organisationUnits.json?fields=id,displayName&paging=false",
    }
    for resource, url in endpoints.items():
        try:
            res = session.get(url, timeout=30)
            if res.ok:
                items = res.json().get(resource, [])
                for item in items:
                    meta_map[item["id"]] = item.get("displayName", item["id"])
        except Exception:
            pass
    return meta_map

def get_data_dhis2(web: str, username: str, password: str, idprogram: list[str], idou: list[str]) -> pd.DataFrame:
    base_url = web.rsplit("/api/", 1)[0].rstrip("/")
    endpoint = f"{base_url}/api/trackedEntityInstances.json"

    session = requests.Session()
    session.auth = (username, password)
    session.headers.update({"Accept": "application/json", "User-Agent": "DHIS2-Python-Script/1.0"})

    retries = Retry(total=5, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    meta_map = build_dhis2_metadata_map(session, base_url)
    all_records = []

    for prog, ou in product(idprogram, idou):
        page = 1
        page_size = 100

        while True:
            params = {
                "program": prog,
                "ou": ou,
                "ouMode": "DESCENDANTS",
                "pageSize": page_size,
                "page": page,
                "totalPages": "true",
                "fields": "trackedEntityInstance,orgUnit,created,lastUpdated,attributes[attribute,displayName,value],enrollments[enrollment,program,orgUnit,enrolledAt,occurredAt,status,events[event,programStage,occurredAt,status,dataValues[dataElement,value]]]",
            }
            try:
                response = session.get(endpoint, params=params, timeout=45)
                if not response.ok:
                    break

                data = response.json()
                instances = data.get("trackedEntityInstances", [])
                if not instances:
                    break

                for instance in instances:
                    ou_id = instance.get("orgUnit")
                    ou_name = meta_map.get(ou_id, ou_id)
                    prog_name = meta_map.get(prog, prog)

                    row = {
                        "program_id": prog,
                        "program_name": prog_name,
                        "trackedEntityInstance": instance.get("trackedEntityInstance"),
                        "orgUnit_id": ou_id,
                        "Org Unit Name": ou_name,
                        "created": instance.get("created"),
                        "lastUpdated": instance.get("lastUpdated"),
                    }

                    for attr in instance.get("attributes", []):
                        attr_id = attr.get("attribute")
                        col_name = attr.get("displayName") or meta_map.get(attr_id) or attr_id
                        row[col_name] = attr.get("value")

                    for enrollment in instance.get("enrollments", []):
                        if enrollment.get("program") == prog:
                            row["enrollment_date"] = enrollment.get("enrolledAt")
                            row["enrollment_status"] = enrollment.get("status")

                            for event in enrollment.get("events", []):
                                stage_id = event.get("programStage")
                                stage_name = meta_map.get(stage_id, f"Stage_{stage_id}")

                                for dv in event.get("dataValues", []):
                                    de_id = dv.get("dataElement")
                                    de_name = meta_map.get(de_id, f"Element_{de_id}")
                                    col_key = f"[{stage_name}] {de_name}"
                                    row[col_key] = dv.get("value")

                    all_records.append(row)

                pager = data.get("pager", {})
                if page >= pager.get("pageCount", 1):
                    break
                page += 1
            except Exception:
                break

    session.close()
    return pd.DataFrame(all_records)

def convert_df_to_excel_bytes(df: pd.DataFrame) -> bytes:
    output = io.BytesIO()
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    header_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
    data_font = Font(name="Segoe UI", size=10)
    alt_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
    thin_border = Border(left=Side(style="thin", color="D9D9D9"), right=Side(style="thin", color="D9D9D9"), top=Side(style="thin", color="D9D9D9"), bottom=Side(style="thin", color="D9D9D9"))

    group_col = "program_name" if "program_name" in df.columns else "program_id"
    for prog_label, prog_df in df.groupby(group_col, dropna=False):
        sheet_title = str(prog_label)[:30] if pd.notna(prog_label) else "Unknown_Program"
        ws = wb.create_sheet(title=sheet_title)
        ws.views.sheetView[0].showGridLines = True

        prog_df_clean = prog_df.dropna(how="all", axis=1)
        headers = list(prog_df_clean.columns)
        ws.append(headers)

        for col_idx in range(1, len(headers) + 1):
            cell = ws.cell(row=1, column=col_idx)
            cell.fill, cell.font = header_fill, header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        for row_idx, record in enumerate(prog_df_clean.to_dict(orient="records"), start=2):
            ws.append([record.get(col, "") for col in headers])
            is_even = row_idx % 2 == 0
            for col_idx in range(1, len(headers) + 1):
                cell = ws.cell(row=row_idx, column=col_idx)
                cell.font, cell.border = data_font, thin_border
                cell.alignment = Alignment(vertical="center")
                if is_even: cell.fill = alt_fill

        ws.freeze_panes = "A2"
        for col in ws.columns:
            max_len = max(len(str(cell.value or "")) for cell in col)
            col_letter = get_column_letter(col[0].column)
            ws.column_dimensions[col_letter].width = min(max(max_len + 4, 12), 45)

    wb.save(output)
    return output.getvalue()


# Define Mappings
PROGRAM_MAP = {
    "Registration & Screening": "UZF0HrTlps0",
    "Diagnostic Evaluation": "GvywHD6crky",
    "TB Case Surveillance": "Lt6P15ps7f6",
    "TB Contact Investigation TPT": "cQsXTtAJ3HW"
}

TOWNSHIP_OU_MAP = {
    "HLG (Hlaing)": ["KqjORlUe8Yc"],
    "KMD (Kyeemyindaing)": ["rTTCKrLpxTB"],
    "SDG (Dagon Myothit South)": ["XHz6CPxTAbR"],
    "TGG (Thingangyun)": ["aBfPB9AwbF5"],
    "SOK (South Okkalapa)": ["OeZsFpNKLP5"],
    "MYG (Mayangone)": ["mPwLv1cjror"],
    "SPT (Shwepyithar)": ["aMAEOgli6W8"]
}



In [3]:
prog_ids = list(PROGRAM_MAP.values())
prog_ids

['UZF0HrTlps0', 'GvywHD6crky', 'Lt6P15ps7f6', 'cQsXTtAJ3HW']

In [4]:
ou_ids = [id for ids in TOWNSHIP_OU_MAP.values() for id in ids]
ou_ids

['KqjORlUe8Yc',
 'rTTCKrLpxTB',
 'XHz6CPxTAbR',
 'aBfPB9AwbF5',
 'OeZsFpNKLP5',
 'mPwLv1cjror',
 'aMAEOgli6W8']

In [5]:
web_url = "https://hmistraining.mm.dhis2.net/train"
username = "Ygn_NTP1"
password = "District@1"

In [6]:
df_result = get_data_dhis2(web_url, username, password, prog_ids, ou_ids)

In [7]:
df_result

,program_id,program_name,trackedEntityInstance,orgUnit_id,Org Unit Name,created,lastUpdated,Nationality,Home Address,GEN - Name,...,[Screening & Testing] TB-HH - Active TB testing,[Screening & Testing] CXR screening date,[Screening & Testing] Enroll to Diagnostic Evaluation,[Screening & Testing] TB CS - Risk factor smoking,[Screening & Testing] TB CS - Risk factor diabetes,[Screening & Testing] Referral organization,[Screening & Testing] Referral activity,[Screening & Testing] CXR screening done,[Screening & Testing] TB CS - HIV infection,[Screening & Testing] Previous TB History
0,UZF0HrTlps0,Registration and Screening,h29C0X4VZ6v,KqjORlUe8Yc,12. YTP/MATA (Hlaing),2026-06-02T05:13:41.473,2026-06-04T05:17:31.859,National,"46, Thayat Myaing St",Thein Htay Aung,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,UZF0HrTlps0,Registration and Screening,zySTcqaFVh8,KqjORlUe8Yc,12. YTP/MATA (Hlaing),2026-06-02T05:23:33.088,2026-06-04T05:17:54.397,National,"99, Pazuntaung 3st,",Min Min Tun,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,UZF0HrTlps0,Registration and Screening,ZFerVs0glZL,KqjORlUe8Yc,12. YTP/MATA (Hlaing),2026-06-02T05:26:21.864,2026-06-04T05:18:59.291,National,"Zayyamingalar 2st,\n",Ha Li Mar B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,UZF0HrTlps0,Registration and Screening,pT1gXZAmImn,KqjORlUe8Yc,12. YTP/MATA (Hlaing),2026-06-02T05:28:51.847,2026-06-04T05:19:44.128,National,"No.142, ThuKhaMyaing 2st,",Kyaw Soe Min,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,UZF0HrTlps0,Registration and Screening,UyxHvIrEdsG,KqjORlUe8Yc,12. YTP/MATA (Hlaing),2026-06-02T05:32:22.607,2026-06-04T05:20:24.091,National,"No.162, Aung Chan Thar St,",Phyu Phyu Win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,Lt6P15ps7f6,TB Case Surveillance,xYpfsEjE5B2,AQpmqMcXgPN,12. MATA (Dagon Myothit (South)),2026-09-12T07:41:15.289,2026-09-12T07:47:59.801,National,NaN,lone lay,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
421,Lt6P15ps7f6,TB Case Surveillance,rsrnxQ0iFeA,rTTCKrLpxTB,12. YTP/MATA (Kyeemyindaing),2026-09-15T07:33:47.161,2026-09-15T07:40:20.787,National,NaN,TEST_TPS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
422,Lt6P15ps7f6,TB Case Surveillance,ueaJQJ3ZMYq,OeZsFpNKLP5,12. YTP/MMA (South Okkalapa),2026-08-15T07:17:03.011,2026-08-15T09:40:36.754,National,Lanmadaw\n304,Thet Ko,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423,Lt6P15ps7f6,TB Case Surveillance,MeHZdcgLuZw,mPwLv1cjror,12. YTP/MMA (Mayangone),2026-08-16T03:04:55.731,2026-08-16T04:08:05.801,National,NaN,Mingalar,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
print(df_result.columns.to_list())

['program_id', 'program_name', 'trackedEntityInstance', 'orgUnit_id', 'Org Unit Name', 'created', 'lastUpdated', 'Nationality', 'Home Address', 'GEN - Name', 'Father Name', 'District', 'GEN - Date of birth', 'Age', 'Ward', 'Unique ID (UPI)', 'GEN - Sex', 'Ward / Village tract', 'NRC No.', 'GEN - Contact phone number (local)', 'Township (T)', 'Region/State', 'enrollment_date', 'enrollment_status', '[TB Screening] Loss of appetite', '[TB Screening] TB CS - Risk factor alcohol', '[TB Screening] CXR result category', '[TB Screening] TB CS - Risk factor undernourishment', '[TB Screening] Cough more than 2 weeks', '[TB Screening] Chest pain', '[TB Screening] CXR screening date', '[TB Screening] TB CS - Risk factor smoking', '[TB Screening] TB CS - Risk factor diabetes', '[TB Screening] CXR screening facility type', '[TB Screening] Referral organization', '[TB Screening] Referral activity', '[TB Screening] Type of CXR', '[TB Screening] CXR screening done', '[TB Screening] CXR result', '[TB Sc

In [10]:
def convert_df_to_excel_bytes(df: pd.DataFrame) -> bytes:
    output = io.BytesIO()
    wb = openpyxl.Workbook()
    wb.remove(wb.active)  # Remove default active sheet

    # Define Column Selection for YgnTBPro
    SelectedColumnList = [
        'Org Unit Name', 'created', 'lastUpdated', 'Nationality', 'Home Address', 
        'GEN - Name', 'Father Name', 'District', 'GEN - Date of birth', 'Age', 
        'Ward', 'Unique ID (UPI)', 'GEN - Sex', 'Ward / Village tract', 'NRC No.', 
        'GEN - Contact phone number (local)', 'Township (T)', 'Region/State', 
        'enrollment_date', 'enrollment_status', '[TB Screening] Loss of appetite', 
        '[TB Screening] TB CS - Risk factor alcohol', '[TB Screening] CXR result category', 
        '[TB Screening] TB CS - Risk factor undernourishment', '[TB Screening] Cough more than 2 weeks', 
        '[TB Screening] Chest pain', '[TB Screening] CXR screening date', 
        '[TB Screening] TB CS - Risk factor smoking', '[TB Screening] TB CS - Risk factor diabetes', 
        '[TB Screening] CXR screening facility type', '[TB Screening] Referral organization', 
        '[TB Screening] Referral activity'
    ]

    # Style Definitions
    header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    header_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
    data_font = Font(name="Segoe UI", size=10)
    alt_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
    thin_border = Border(
        left=Side(style="thin", color="D9D9D9"),
        right=Side(style="thin", color="D9D9D9"),
        top=Side(style="thin", color="D9D9D9"),
        bottom=Side(style="thin", color="D9D9D9")
    )

    def _format_and_populate_sheet(ws, sheet_df: pd.DataFrame):
        """Helper function to format headers, cells, zebra-striping, and auto-fit columns."""
        ws.views.sheetView[0].showGridLines = True
        
        # Clean up columns that are completely empty
        clean_df = sheet_df.dropna(how="all", axis=1)
        headers = list(clean_df.columns)
        
        if not headers:
            return

        # Append Header
        ws.append(headers)
        for col_idx in range(1, len(headers) + 1):
            cell = ws.cell(row=1, column=col_idx)
            cell.fill, cell.font = header_fill, header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        # Append Data Rows & Apply Styling
        for row_idx, record in enumerate(clean_df.to_dict(orient="records"), start=2):
            ws.append([record.get(col, "") for col in headers])
            is_even = row_idx % 2 == 0
            for col_idx in range(1, len(headers) + 1):
                cell = ws.cell(row=row_idx, column=col_idx)
                cell.font, cell.border = data_font, thin_border
                cell.alignment = Alignment(vertical="center")
                if is_even:
                    cell.fill = alt_fill

        # Freeze Headers & Set Column Widths
        ws.freeze_panes = "A2"
        for col in ws.columns:
            max_len = max(len(str(cell.value or "")) for cell in col)
            col_letter = get_column_letter(col[0].column)
            ws.column_dimensions[col_letter].width = min(max(max_len + 4, 12), 45)

    # 1. ADD CONSOLIDATED SHEET (All Data)
    ws_consolidated = wb.create_sheet(title="Consolidated")
    _format_and_populate_sheet(ws_consolidated, df)

    # 2. ADD YgnTBPro SHEET (Filter Selected Columns that exist in df)
    existing_cols = [col for col in SelectedColumnList if col in df.columns]
    df_ygntbpro = df[existing_cols] if existing_cols else pd.DataFrame()
    ws_ygntbpro = wb.create_sheet(title="YgnTBPro")
    _format_and_populate_sheet(ws_ygntbpro, df_ygntbpro)

    # 3. ADD PROGRAM-SPECIFIC SHEETS
    group_col = "program_name" if "program_name" in df.columns else "program_id"
    if group_col in df.columns:
        for prog_label, prog_df in df.groupby(group_col, dropna=False):
            # Clean sheet title (Max 30 chars, remove illegal characters)
            sheet_title = str(prog_label)[:30] if pd.notna(prog_label) else "Unknown_Program"
            for char in [":", "\\", "/", "?", "*", "[", "]"]:
                sheet_title = sheet_title.replace(char, "_")
            
            ws_prog = wb.create_sheet(title=sheet_title)
            _format_and_populate_sheet(ws_prog, prog_df)

    wb.save(output)
    return output.getvalue()

In [11]:
convert_df_to_excel_bytes(df_result)

b'PK\x03\x04\x14\x00\x00\x00\x08\x00E~8]F\xc7MH\x95\x00\x00\x00\xcd\x00\x00\x00\x10\x00\x00\x00docProps/app.xmlM\xcfM\x0b\xc20\x0c\x06\xe0\xbfRv\xb7\x99\x8a\x1e\xa4\x0eD=\x8a\x9e\xbc\xcf.u\x85\xb6)m\x84\xfa\xef\xed\x04?nyy\xc8\x1b\xa2.\x89"&\xb6\x98E\xf1.\xe4m32\xc7\r@\xd6#\xfa>\xcb\xca\xa1\x8a\xa1\xe4{\xae1\xdd\x81\x8c\xb1\x1a\x0f\xa4\x1f\x1e\x03\xc3\xa2m\xd7\x80\x851\x0c8\xcc\xe2\xb7\xb0\xe9\xd4.Fgu\xcf\x96Bw\xb2:Q&\xc3\xe2X4:\xb1\'\x1f\xab\xdc\x1c\n\x10\xe7z%>\x8b\x13K9\x97+\x05\xff\x8bS\xcb\x15S\x9e\xe6\xcao\xfcd\x05\xbf\x07\xba\x17PK\x03\x04\x14\x00\x00\x00\x08\x00E~8],\x19\xa4\xa2\xee\x00\x00\x00\xcb\x01\x00\x00\x11\x00\x00\x00docProps/core.xml\x95\x91\xc1N\xc30\x0c\x86_e\xca\xbdu\x9a\xa1\x89E]. N !1\t\xc4-J\xbc-\xa2i\xa3\xc4\xa8\xdd\xdb\x93\x96\xad\x1b\x82\x0b\xc7\xf8\xff\xfc\xd9Vj\x13\xa4\xe9">\xc7.`$\x87i1\xf8\xa6M\xd2\x84\r;\x10\x05\t\x90\xcc\x01\xbdNe&\xda\x1c\xee\xba\xe85\xe5g\xdcC\xd0\xe6C\xef\x11\x04\xe7+\xf0H\xdaj\xd20\n\x8b0\x1b\xd9Ii\xcd\xac\x0c\x9f\xb1\x99\x04\xd6\x00

In [12]:
df_result1 = df_result.reindex(sorted(df_result.columns, reverse=True), axis=1)  # Sort columns in reverse alphabetical order (Z -> A)

In [13]:
df_result1

,trackedEntityInstance,program_name,program_id,orgUnit_id,lastUpdated,enrollment_status,enrollment_date,created,[TB Screening] Year of last TB Treatment,[TB Screening] Weight loss,...,Home Address,GEN - Sex,GEN - Name,GEN - Date of birth is estimated,GEN - Date of birth,GEN - Contact phone number (local),Father Name,District,Country of origin,Age
0,h29C0X4VZ6v,Registration and Screening,UZF0HrTlps0,KqjORlUe8Yc,2026-06-04T05:17:31.859,ACTIVE,None,2026-06-02T05:13:41.473,NaN,NaN,...,"46, Thayat Myaing St",MALE,Thein Htay Aung,NaN,2004-06-02,09753122470,U Hla Moe,Mayangon,NaN,21
1,zySTcqaFVh8,Registration and Screening,UZF0HrTlps0,KqjORlUe8Yc,2026-06-04T05:17:54.397,ACTIVE,None,2026-06-02T05:23:33.088,NaN,NaN,...,"99, Pazuntaung 3st,",MALE,Min Min Tun,NaN,1995-01-05,09668009524,U Aye Naing,Mayangon,NaN,31
2,ZFerVs0glZL,Registration and Screening,UZF0HrTlps0,KqjORlUe8Yc,2026-06-04T05:18:59.291,ACTIVE,None,2026-06-02T05:26:21.864,NaN,NaN,...,"Zayyamingalar 2st,\n",FEMALE,Ha Li Mar B,NaN,1981-02-26,09972977195,U Tha Mar,Mayangon,NaN,45
3,pT1gXZAmImn,Registration and Screening,UZF0HrTlps0,KqjORlUe8Yc,2026-06-04T05:19:44.128,ACTIVE,None,2026-06-02T05:28:51.847,NaN,NaN,...,"No.142, ThuKhaMyaing 2st,",MALE,Kyaw Soe Min,NaN,1991-12-01,0925418958,U Aung Naing,Mayangon,NaN,34
4,UyxHvIrEdsG,Registration and Screening,UZF0HrTlps0,KqjORlUe8Yc,2026-06-04T05:20:24.091,ACTIVE,None,2026-06-02T05:32:22.607,NaN,NaN,...,"No.162, Aung Chan Thar St,",FEMALE,Phyu Phyu Win,NaN,1964-06-01,09xxxxxxx,U Kyaw Min,Mayangon,NaN,62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,xYpfsEjE5B2,TB Case Surveillance,Lt6P15ps7f6,AQpmqMcXgPN,2026-09-12T07:47:59.801,ACTIVE,None,2026-09-12T07:41:15.289,NaN,NaN,...,NaN,FEMALE,lone lay,NaN,1999-01-11,09xxxxxxx,lone lone,Amarapura,NaN,27
421,rsrnxQ0iFeA,TB Case Surveillance,Lt6P15ps7f6,rTTCKrLpxTB,2026-09-15T07:40:20.787,ACTIVE,None,2026-09-15T07:33:47.161,NaN,NaN,...,NaN,MALE,TEST_TPS,NaN,2000-01-15,09xxxxxxxxx,TPS FATHER,Kamayut,NaN,26
422,ueaJQJ3ZMYq,TB Case Surveillance,Lt6P15ps7f6,OeZsFpNKLP5,2026-08-15T09:40:36.754,COMPLETED,None,2026-08-15T07:17:03.011,NaN,NaN,...,Lanmadaw\n304,MALE,Thet Ko,NaN,2000-12-19,09976056279,Father Thet Ko,Monywa,NaN,25
423,MeHZdcgLuZw,TB Case Surveillance,Lt6P15ps7f6,mPwLv1cjror,2026-08-16T04:08:05.801,ACTIVE,None,2026-08-16T03:04:55.731,NaN,NaN,...,NaN,MALE,Mingalar,NaN,2000-01-01,09xxxxxxxxx,father mingalar,Aunglan,NaN,26


In [14]:
df_result2 = df_result1.reindex(sorted(df_result1.columns), axis=1)  # Sort columns alphabetically (A -> Z)
df_result2

,Age,Country of origin,District,Father Name,GEN - Contact phone number (local),GEN - Date of birth,GEN - Date of birth is estimated,GEN - Name,GEN - Sex,Home Address,...,[TB Screening] Weight loss,[TB Screening] Year of last TB Treatment,created,enrollment_date,enrollment_status,lastUpdated,orgUnit_id,program_id,program_name,trackedEntityInstance
0,21,NaN,Mayangon,U Hla Moe,09753122470,2004-06-02,NaN,Thein Htay Aung,MALE,"46, Thayat Myaing St",...,NaN,NaN,2026-06-02T05:13:41.473,None,ACTIVE,2026-06-04T05:17:31.859,KqjORlUe8Yc,UZF0HrTlps0,Registration and Screening,h29C0X4VZ6v
1,31,NaN,Mayangon,U Aye Naing,09668009524,1995-01-05,NaN,Min Min Tun,MALE,"99, Pazuntaung 3st,",...,NaN,NaN,2026-06-02T05:23:33.088,None,ACTIVE,2026-06-04T05:17:54.397,KqjORlUe8Yc,UZF0HrTlps0,Registration and Screening,zySTcqaFVh8
2,45,NaN,Mayangon,U Tha Mar,09972977195,1981-02-26,NaN,Ha Li Mar B,FEMALE,"Zayyamingalar 2st,\n",...,NaN,NaN,2026-06-02T05:26:21.864,None,ACTIVE,2026-06-04T05:18:59.291,KqjORlUe8Yc,UZF0HrTlps0,Registration and Screening,ZFerVs0glZL
3,34,NaN,Mayangon,U Aung Naing,0925418958,1991-12-01,NaN,Kyaw Soe Min,MALE,"No.142, ThuKhaMyaing 2st,",...,NaN,NaN,2026-06-02T05:28:51.847,None,ACTIVE,2026-06-04T05:19:44.128,KqjORlUe8Yc,UZF0HrTlps0,Registration and Screening,pT1gXZAmImn
4,62,NaN,Mayangon,U Kyaw Min,09xxxxxxx,1964-06-01,NaN,Phyu Phyu Win,FEMALE,"No.162, Aung Chan Thar St,",...,NaN,NaN,2026-06-02T05:32:22.607,None,ACTIVE,2026-06-04T05:20:24.091,KqjORlUe8Yc,UZF0HrTlps0,Registration and Screening,UyxHvIrEdsG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,27,NaN,Amarapura,lone lone,09xxxxxxx,1999-01-11,NaN,lone lay,FEMALE,NaN,...,NaN,NaN,2026-09-12T07:41:15.289,None,ACTIVE,2026-09-12T07:47:59.801,AQpmqMcXgPN,Lt6P15ps7f6,TB Case Surveillance,xYpfsEjE5B2
421,26,NaN,Kamayut,TPS FATHER,09xxxxxxxxx,2000-01-15,NaN,TEST_TPS,MALE,NaN,...,NaN,NaN,2026-09-15T07:33:47.161,None,ACTIVE,2026-09-15T07:40:20.787,rTTCKrLpxTB,Lt6P15ps7f6,TB Case Surveillance,rsrnxQ0iFeA
422,25,NaN,Monywa,Father Thet Ko,09976056279,2000-12-19,NaN,Thet Ko,MALE,Lanmadaw\n304,...,NaN,NaN,2026-08-15T07:17:03.011,None,COMPLETED,2026-08-15T09:40:36.754,OeZsFpNKLP5,Lt6P15ps7f6,TB Case Surveillance,ueaJQJ3ZMYq
423,26,NaN,Aunglan,father mingalar,09xxxxxxxxx,2000-01-01,NaN,Mingalar,MALE,NaN,...,NaN,NaN,2026-08-16T03:04:55.731,None,ACTIVE,2026-08-16T04:08:05.801,mPwLv1cjror,Lt6P15ps7f6,TB Case Surveillance,MeHZdcgLuZw


In [16]:
print(list(df_result2.columns))

['Age', 'Country of origin', 'District', 'Father Name', 'GEN - Contact phone number (local)', 'GEN - Date of birth', 'GEN - Date of birth is estimated', 'GEN - Name', 'GEN - Sex', 'Home Address', 'NRC No.', 'Nationality', 'Org Unit Name', 'Passport number', 'Region/State', 'Relationship with index', 'Township (T)', 'Tracker Id', 'Unique ID (UPI)', 'Unique ID (UPI) - Index Case', 'Village', 'Ward', 'Ward / Village tract', '[01. Screening and Case History] Activity Notes', '[01. Screening and Case History] Age (at screening)', '[01. Screening and Case History] Any TB drug resistance history?', '[01. Screening and Case History] BMI', '[01. Screening and Case History] Breathlessness', '[01. Screening and Case History] CXR result', '[01. Screening and Case History] CXR result category', '[01. Screening and Case History] CXR screening date', '[01. Screening and Case History] CXR screening done', '[01. Screening and Case History] CXR screening facility type', '[01. Screening and Case History]

In [ ]:
'lastUpdated','Org Unit Name','Region/State','District','Township (T)','Ward','Unique ID (UPI)','GEN - Name', 'GEN - Sex','Age','GEN - Contact phone number (local)'

In [ ]:
['Age', 'Country of origin', 'District', 'Father Name', 'GEN - Contact phone number (local)', 'GEN - Date of birth', 'GEN - Date of birth is estimated', 'GEN - Name', 'GEN - Sex', 'Home Address', 'NRC No.', 'Nationality', 'Org Unit Name', 'Passport number', 'Region/State', 'Relationship with index', 'Township (T)', 'Tracker Id', 'Unique ID (UPI)', 'Unique ID (UPI) - Index Case', 'Village', 'Ward', 'Ward / Village tract', '[01. Screening and Case History] Activity Notes', '[01. Screening and Case History] Age (at screening)', '[01. Screening and Case History] Any TB drug resistance history?', '[01. Screening and Case History] BMI', '[01. Screening and Case History] Breathlessness', '[01. Screening and Case History] CXR result', '[01. Screening and Case History] CXR result category', '[01. Screening and Case History] CXR screening date', '[01. Screening and Case History] CXR screening done', '[01. Screening and Case History] CXR screening facility type', '[01. Screening and Case History] Chest pain', '[01. Screening and Case History] Cough more than 2 weeks', '[01. Screening and Case History] Current activity', '[01. Screening and Case History] Fatigue and Tiredness', '[01. Screening and Case History] Fever more than 2 weeks', '[01. Screening and Case History] Haemoptysis', '[01. Screening and Case History] Healthcare worker population', '[01. Screening and Case History] Height (in inches)', '[01. Screening and Case History] Household Contact', '[01. Screening and Case History] Loss of appetite', '[01. Screening and Case History] Migrant population', '[01. Screening and Case History] Night sweats', '[01. Screening and Case History] No symptoms related with TB', '[01. Screening and Case History] Number of previous TB episodes', '[01. Screening and Case History] Other symptoms related with TB', '[01. Screening and Case History] Previous TB History', '[01. Screening and Case History] Previous TB regimen', '[01. Screening and Case History] Referral activity', '[01. Screening and Case History] Referral organization', '[01. Screening and Case History] TB CS - HIV infection', '[01. Screening and Case History] TB CS - HIV status date', '[01. Screening and Case History] TB CS - Registration - Type of patient in last TB Treatment', '[01. Screening and Case History] TB CS - Risk factor alcohol', '[01. Screening and Case History] TB CS - Risk factor diabetes', '[01. Screening and Case History] TB CS - Risk factor smoking', '[01. Screening and Case History] TB CS - Risk factor undernourishment', '[01. Screening and Case History] Type of CXR', '[01. Screening and Case History] Weight (in kgs)', '[01. Screening and Case History] Weight loss', '[01. Screening and Case History] Year of last TB Treatment', '[02. Examination Request and Results] Date of Sample Received at Laboratory', '[02. Examination Request and Results] Date of specimen collected', '[02. Examination Request and Results] Name of laboratory examination facility', '[02. Examination Request and Results] Reason for Xpert MTB/XDR request', '[02. Examination Request and Results] Reject sample', '[02. Examination Request and Results] Result data available for dispatch', '[02. Examination Request and Results] Result date', '[02. Examination Request and Results] Result dispatch date', '[02. Examination Request and Results] Samples received (number)', '[02. Examination Request and Results] Specimen type', '[02. Examination Request and Results] TB CS - Resistance classification', '[02. Examination Request and Results] TB-LAB - Sample ID', '[02. Examination Request and Results] TB-LAB - Xpert MTB/RIF Ultra', '[02. Examination Request and Results] TB-LAB - Xpert MTB/RIF Ultra result', '[02. Examination Request and Results] Visual appearance', '[02. Examination Request and Results] Xpert MTB/XDR', '[02. Examination Request and Results] Xpert MTB/XDR (Examination)', '[02. Examination Request and Results] Xpert MTB/XDR Result for Am', '[02. Examination Request and Results] Xpert MTB/XDR Result for Cm', '[02. Examination Request and Results] Xpert MTB/XDR Result for Eto', '[02. Examination Request and Results] Xpert MTB/XDR Result for Fq', '[02. Examination Request and Results] Xpert MTB/XDR Result for INH', '[02. Examination Request and Results] Xpert MTB/XDR Result for Km', '[03. TB Diagnosis] Enroll to TB Case Surveillance Program', '[03. TB Diagnosis] TB CS - Confirmation method', '[03. TB Diagnosis] TB CS - Diagnosis date', '[03. TB Diagnosis] TB CS - Diagnostic note', '[03. TB Diagnosis] TB CS - Manually assigned resistance classification', '[03. TB Diagnosis] TB CS - Reassign resistance classification', '[03. TB Diagnosis] TB CS - Resistance classification', '[03. TB Diagnosis] TB CS - Site of disease', '[03. TB Diagnosis] TB CS Diagnosis', '[03. TB Diagnosis] Type of TB patient', '[1. Screening, Diagnosis and Notification] Age (at screening)', '[1. Screening, Diagnosis and Notification] Any TB drug resistance history?', '[1. Screening, Diagnosis and Notification] BMI', '[1. Screening, Diagnosis and Notification] Breathlessness', '[1. Screening, Diagnosis and Notification] CXR result', '[1. Screening, Diagnosis and Notification] CXR result category', '[1. Screening, Diagnosis and Notification] CXR screening date', '[1. Screening, Diagnosis and Notification] CXR screening done', '[1. Screening, Diagnosis and Notification] CXR screening facility type', '[1. Screening, Diagnosis and Notification] Cough more than 2 weeks', '[1. Screening, Diagnosis and Notification] Current activity', '[1. Screening, Diagnosis and Notification] Fatigue and Tiredness', '[1. Screening, Diagnosis and Notification] Fever more than 2 weeks', '[1. Screening, Diagnosis and Notification] Haemoptysis', '[1. Screening, Diagnosis and Notification] Healthcare worker population', '[1. Screening, Diagnosis and Notification] Height (in inches)', '[1. Screening, Diagnosis and Notification] Loss of appetite', '[1. Screening, Diagnosis and Notification] Migrant population', '[1. Screening, Diagnosis and Notification] Night sweats', '[1. Screening, Diagnosis and Notification] Number of previous TB episodes', '[1. Screening, Diagnosis and Notification] Other symptoms related with TB', '[1. Screening, Diagnosis and Notification] Previous TB History', '[1. Screening, Diagnosis and Notification] Previous TB regimen', '[1. Screening, Diagnosis and Notification] Referral activity', '[1. Screening, Diagnosis and Notification] Referral organization', '[1. Screening, Diagnosis and Notification] Started on Treatment', '[1. Screening, Diagnosis and Notification] TB CS - ART patient ID', '[1. Screening, Diagnosis and Notification] TB CS - Case notification', '[1. Screening, Diagnosis and Notification] TB CS - Confirmation method', '[1. Screening, Diagnosis and Notification] TB CS - Diagnosis date', '[1. Screening, Diagnosis and Notification] TB CS - Diagnostic note', '[1. Screening, Diagnosis and Notification] TB CS - HIV infection', '[1. Screening, Diagnosis and Notification] TB CS - HIV status date', '[1. Screening, Diagnosis and Notification] TB CS - Manually assigned resistance classification', '[1. Screening, Diagnosis and Notification] TB CS - Notification date', '[1. Screening, Diagnosis and Notification] TB CS - Reassign resistance classification', '[1. Screening, Diagnosis and Notification] TB CS - Registration - Type of patient in last TB Treatment', '[1. Screening, Diagnosis and Notification] TB CS - Resistance classification', '[1. Screening, Diagnosis and Notification] TB CS - Risk factor alcohol', '[1. Screening, Diagnosis and Notification] TB CS - Risk factor diabetes', '[1. Screening, Diagnosis and Notification] TB CS - Risk factor smoking', '[1. Screening, Diagnosis and Notification] TB CS - Risk factor undernourishment', '[1. Screening, Diagnosis and Notification] TB CS - Site of disease', '[1. Screening, Diagnosis and Notification] Type of CXR', '[1. Screening, Diagnosis and Notification] Type of TB patient', '[1. Screening, Diagnosis and Notification] Weight (in kgs)', '[1. Screening, Diagnosis and Notification] Weight loss', '[1. Screening, Diagnosis and Notification] Year of last TB Treatment', '[2. TB Treatment] TB CS - Diagnosis date', '[2. TB Treatment] TB CS - First-line treatment regimen composition', '[2. TB Treatment] TB CS - First-line treatment start date', '[2. TB Treatment] TB CS - Manually assigned resistance classification', '[2. TB Treatment] TB CS - Outcome due date', '[2. TB Treatment] TB CS - Reassign resistance classification', '[2. TB Treatment] TB CS - Resistance at diagnosis', '[2. TB Treatment] TB CS - Resistance classification', '[2. TB Treatment] TB CS - Treatment initiation delay (days)', '[2. TB Treatment] TB CS - Treatment regimen', '[4. Outcome] TB CS - Treatment outcome', '[4. Outcome] TB CS - Treatment outcome delay (weeks)', '[Screening & Testing] BMI', '[Screening & Testing] CXR result category', '[Screening & Testing] CXR screening date', '[Screening & Testing] CXR screening done', '[Screening & Testing] Enroll to Diagnostic Evaluation', '[Screening & Testing] Height (in inches)', '[Screening & Testing] Previous TB History', '[Screening & Testing] Referral activity', '[Screening & Testing] Referral organization', '[Screening & Testing] TB CS - HIV infection', '[Screening & Testing] TB CS - Risk factor alcohol', '[Screening & Testing] TB CS - Risk factor diabetes', '[Screening & Testing] TB CS - Risk factor smoking', '[Screening & Testing] TB CS - Risk factor undernourishment', '[Screening & Testing] TB-HH - Active TB test result', '[Screening & Testing] TB-HH - Active TB testing', '[Screening & Testing] TB-HH - Screening', '[Screening & Testing] Weight (in kgs)', '[TB Screening] Age (at screening)', '[TB Screening] Any TB drug resistance history?', '[TB Screening] BMI', '[TB Screening] Breathlessness', '[TB Screening] CXR result', '[TB Screening] CXR result category', '[TB Screening] CXR screening date', '[TB Screening] CXR screening done', '[TB Screening] CXR screening facility type', '[TB Screening] Chest pain', '[TB Screening] Cough more than 2 weeks', '[TB Screening] Current activity', '[TB Screening] Enroll to Diagnostic Evaluation', '[TB Screening] Fatigue and Tiredness', '[TB Screening] Fever more than 2 weeks', '[TB Screening] Haemoptysis', '[TB Screening] Healthcare worker population', '[TB Screening] Height (in inches)', '[TB Screening] Household Contact', '[TB Screening] Loss of appetite', '[TB Screening] Migrant population', '[TB Screening] Night sweats', '[TB Screening] No symptoms related with TB', '[TB Screening] Number of previous TB episodes', '[TB Screening] Other referral organisation (Specify)', '[TB Screening] Other symptoms related with TB', '[TB Screening] Previous TB History', '[TB Screening] Previous TB regimen', '[TB Screening] Referral activity', '[TB Screening] Referral organization', '[TB Screening] Specify other symptoms', '[TB Screening] TB CS - HIV infection', '[TB Screening] TB CS - HIV status date', '[TB Screening] TB CS - Registration - Type of patient in last TB Treatment', '[TB Screening] TB CS - Risk factor alcohol', '[TB Screening] TB CS - Risk factor diabetes', '[TB Screening] TB CS - Risk factor smoking', '[TB Screening] TB CS - Risk factor undernourishment', '[TB Screening] Type of CXR', '[TB Screening] Weight (in kgs)', '[TB Screening] Weight loss', '[TB Screening] Year of last TB Treatment', 'created', 'enrollment_date', 'enrollment_status', 'lastUpdated', 'orgUnit_id', 'program_id', 'program_name', 'trackedEntityInstance']

In [ ]:
from itertools import product
import io
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
import streamlit as st
from urllib3.util import Retry

# --- Backend DHIS2 Processing Functions ---
def build_dhis2_metadata_map(session: requests.Session, base_url: str) -> dict[str, str]:
    meta_map = {}
    endpoints = {
        "trackedEntityAttributes": f"{base_url}/api/trackedEntityAttributes.json?fields=id,displayName&paging=false",
        "dataElements": f"{base_url}/api/dataElements.json?fields=id,displayName&paging=false",
        "programStages": f"{base_url}/api/programStages.json?fields=id,displayName&paging=false",
        "programs": f"{base_url}/api/programs.json?fields=id,displayName&paging=false",
        "organisationUnits": f"{base_url}/api/organisationUnits.json?fields=id,displayName&paging=false",
    }
    for resource, url in endpoints.items():
        try:
            res = session.get(url, timeout=30)
            if res.ok:
                items = res.json().get(resource, [])
                for item in items:
                    meta_map[item["id"]] = item.get("displayName", item["id"])
        except Exception:
            pass
    return meta_map

def get_data_dhis2(web: str, username: str, password: str, idprogram: list[str], idou: list[str]) -> pd.DataFrame:
    base_url = web.rsplit("/api/", 1)[0].rstrip("/")
    endpoint = f"{base_url}/api/trackedEntityInstances.json"

    session = requests.Session()
    session.auth = (username, password)
    session.headers.update({"Accept": "application/json", "User-Agent": "DHIS2-Python-Script/1.0"})

    retries = Retry(total=5, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    meta_map = build_dhis2_metadata_map(session, base_url)
    all_records = []

    for prog, ou in product(idprogram, idou):
        page = 1
        page_size = 100

        while True:
            params = {
                "program": prog,
                "ou": ou,
                "ouMode": "DESCENDANTS",
                "pageSize": page_size,
                "page": page,
                "totalPages": "true",
                "fields": "trackedEntityInstance,orgUnit,created,lastUpdated,attributes[attribute,displayName,value],enrollments[enrollment,program,orgUnit,enrolledAt,occurredAt,status,events[event,programStage,occurredAt,status,dataValues[dataElement,value]]]",
            }
            try:
                response = session.get(endpoint, params=params, timeout=45)
                if not response.ok:
                    break

                data = response.json()
                instances = data.get("trackedEntityInstances", [])
                if not instances:
                    break

                for instance in instances:
                    ou_id = instance.get("orgUnit")
                    ou_name = meta_map.get(ou_id, ou_id)
                    prog_name = meta_map.get(prog, prog)

                    row = {
                        "program_id": prog,
                        "program_name": prog_name,
                        "trackedEntityInstance": instance.get("trackedEntityInstance"),
                        "orgUnit_id": ou_id,
                        "Org Unit Name": ou_name,
                        "created": instance.get("created"),
                        "lastUpdated": instance.get("lastUpdated"),
                    }

                    for attr in instance.get("attributes", []):
                        attr_id = attr.get("attribute")
                        col_name = attr.get("displayName") or meta_map.get(attr_id) or attr_id
                        row[col_name] = attr.get("value")

                    for enrollment in instance.get("enrollments", []):
                        if enrollment.get("program") == prog:
                            row["enrollment_date"] = enrollment.get("enrolledAt")
                            row["enrollment_status"] = enrollment.get("status")

                            for event in enrollment.get("events", []):
                                stage_id = event.get("programStage")
                                stage_name = meta_map.get(stage_id, f"Stage_{stage_id}")

                                for dv in event.get("dataValues", []):
                                    de_id = dv.get("dataElement")
                                    de_name = meta_map.get(de_id, f"Element_{de_id}")
                                    col_key = f"[{stage_name}] {de_name}"
                                    row[col_key] = dv.get("value")

                    all_records.append(row)

                pager = data.get("pager", {})
                if page >= pager.get("pageCount", 1):
                    break
                page += 1
            except Exception:
                break

    session.close()
    return pd.DataFrame(all_records)

def convert_df_to_excel_bytes(df: pd.DataFrame) -> bytes:
    output = io.BytesIO()
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    header_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
    data_font = Font(name="Segoe UI", size=10)
    alt_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
    thin_border = Border(left=Side(style="thin", color="D9D9D9"), right=Side(style="thin", color="D9D9D9"), top=Side(style="thin", color="D9D9D9"), bottom=Side(style="thin", color="D9D9D9"))

    group_col = "program_name" if "program_name" in df.columns else "program_id"
    for prog_label, prog_df in df.groupby(group_col, dropna=False):
        sheet_title = str(prog_label)[:30] if pd.notna(prog_label) else "Unknown_Program"
        ws = wb.create_sheet(title=sheet_title)
        ws.views.sheetView[0].showGridLines = True

        prog_df_clean = prog_df.dropna(how="all", axis=1)
        headers = list(prog_df_clean.columns)
        ws.append(headers)

        for col_idx in range(1, len(headers) + 1):
            cell = ws.cell(row=1, column=col_idx)
            cell.fill, cell.font = header_fill, header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        for row_idx, record in enumerate(prog_df_clean.to_dict(orient="records"), start=2):
            ws.append([record.get(col, "") for col in headers])
            is_even = row_idx % 2 == 0
            for col_idx in range(1, len(headers) + 1):
                cell = ws.cell(row=row_idx, column=col_idx)
                cell.font, cell.border = data_font, thin_border
                cell.alignment = Alignment(vertical="center")
                if is_even: cell.fill = alt_fill

        ws.freeze_panes = "A2"
        for col in ws.columns:
            max_len = max(len(str(cell.value or "")) for cell in col)
            col_letter = get_column_letter(col[0].column)
            ws.column_dimensions[col_letter].width = min(max(max_len + 4, 12), 45)

    wb.save(output)
    return output.getvalue()

# PROGRAM ID { Registration&Screening : UZF0HrTlps0 , 
#               DiagnosticEvaluation : GvywHD6crky , 
#               TBCaseSurveillence : Lt6P15ps7f6 ,
#               TBContactInvestigationTPT : cQsXTtAJ3HW }

# ORGANISATION UNIT ID { YTPMATA_HLG : KqjORlUe8Yc , 
#                       YTPMATA_KMD : rTTCKrLpxTB ,
#                       YTPMATA_SDG : XHz6CPxTAbR , 
#                       YTPMMA_TGG : MlBn9fEP74R ,
#                       YTPMATA_TGG : aBfPB9AwbF5 ,
#                       YTPMATA_SOK : OeZsFpNKLP5 ,
#                       YTPMATA_MYG : mPwLv1cjror ,
#                       YTPMATA_SPT : aMAEOgli6W8 }

# # --- Streamlit UI Components ---
# st.title("📊 DHIS2 Tracker Exporter")
# st.markdown("Extract tracked entity instances from DHIS2 and download formatted Excel files.")

# with st.form("dhis2_form"):
#     web_url = st.text_input("DHIS2 Base URL", value="https://hmistraining.mm.dhis2.net/train")
#     col1, col2 = st.columns(2)
#     with col1:
#         username = st.text_input("Username", value="Ygn_NTP1")
#     with col2:
#         password = st.text_input("Password", type="password", value="District@1")



#     prog_input = st.text_area("Program IDs (comma-separated)", value="UZF0HrTlps0, GvywHD6crky, Lt6P15ps7f6, cQsXTtAJ3HW")




#     ou_input = st.text_area("Org Unit IDs (comma-separated)", value="KqjORlUe8Yc, rTTCKrLpxTB, XHz6CPxTAbR, MlBn9fEP74R, aBfPB9AwbF5, OeZsFpNKLP5, mPwLv1cjror, aMAEOgli6W8")
    
#     submitted = st.form_submit_button("Extract & Process Data")

# if submitted:
#     prog_ids = [p.strip() for p in prog_input.split(",") if p.strip()]
#     ou_ids = [o.strip() for o in ou_input.split(",") if o.strip()]

#     with st.spinner("Connecting to DHIS2 and extracting data..."):
#         df_result = get_data_dhis2(web_url, username, password, prog_ids, ou_ids)

#     if df_result.empty:
#         st.error("No data extracted. Check your parameters or credentials.")
#     else:
#         st.success(f"Successfully extracted {len(df_result)} records!")
#         excel_data = convert_df_to_excel_bytes(df_result)
        
#         st.download_button(
#             label="💾 Download Excel File (.xlsx)",
#             data=excel_data,
#             file_name="DHIS2_Tracker_Export.xlsx",
#             mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
#             type="primary"
#         )


# --- Streamlit UI Components ---
st.title("📊 DHIS2 Tracker Exporter")
st.markdown("Extract tracked entity instances from DHIS2 and download formatted Excel files.")

# Define Mappings
PROGRAM_MAP = {
    "Registration & Screening": "UZF0HrTlps0",
    "Diagnostic Evaluation": "GvywHD6crky",
    "TB Case Surveillance": "Lt6P15ps7f6",
    "TB Contact Investigation TPT": "cQsXTtAJ3HW"
}

TOWNSHIP_OU_MAP = {
    "HLG (Hlaing)": ["KqjORlUe8Yc"],
    "KMD (Kyeemyindaing)": ["rTTCKrLpxTB"],
    "SDG (Dagon Myothit South)": ["XHz6CPxTAbR"],
    "TGG (Thingangyun)": ["aBfPB9AwbF5"],
    "SOK (South Okkalapa)": ["OeZsFpNKLP5"],
    "MYG (Mayangone)": ["mPwLv1cjror"],
    "SPT (Shwepyithar)": ["aMAEOgli6W8"]
}

with st.form("dhis2_form"):
    web_url = st.text_input("DHIS2 Base URL", value="https://hmistraining.mm.dhis2.net/train")
    col1, col2 = st.columns(2)
    with col1:
        username = st.text_input("Username", value="Ygn_NTP1")
    with col2:
        password = st.text_input("Password", type="password", value="District@1")

    # Program Selection UI
    selected_programs = st.multiselect(
        "Select Programs",
        options=list(PROGRAM_MAP.keys()),
        default=list(PROGRAM_MAP.keys())
    )

    # Township Selection UI
    selected_townships = st.multiselect(
        "Select Townships",
        options=list(TOWNSHIP_OU_MAP.keys()),
        default=list(TOWNSHIP_OU_MAP.keys())
    )

    submitted = st.form_submit_button("Extract & Process Data")

if submitted:
    # Extract selected Program IDs
    prog_ids = [PROGRAM_MAP[p] for p in selected_programs]

    # Extract and flatten selected Township Org Unit IDs
    ou_ids = []
    for township in selected_townships:
        ou_ids.extend(TOWNSHIP_OU_MAP[township])

    if not prog_ids or not ou_ids:
        st.warning("Please select at least one Program and one Township.")
    else:
        with st.spinner("Connecting to DHIS2 and extracting data..."):
            df_result = get_data_dhis2(web_url, username, password, prog_ids, ou_ids)

        if df_result.empty:
            st.error("No data extracted. Check your parameters or credentials.")
        else:
            st.success(f"Successfully extracted {len(df_result)} records!")
            excel_data = convert_df_to_excel_bytes(df_result)
            
            st.download_button(
                label="💾 Download Excel File (.xlsx)",
                data=excel_data,
                file_name="DHIS2_Tracker_Export.xlsx",
                mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
                type="primary"
            )